In [1]:
import sys
!{sys.executable} -m pip install Pillow pillow-heif

--- Logging error ---
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_internal/utils/logging.py", line 177, in emit
    self.console.print(renderable, overflow="ignore", crop=False, style=style)
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_vendor/rich/console.py", line 1673, in print
    extend(render(renderable, render_options))
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_vendor/rich/console.py", line 1305, in render
    for render_output in iter_render:
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_internal/utils/logging.py", line 134, in __rich_console__
    for line in lines:
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_vendor/rich/segment.py", line 249, in split_lines
    for segment in segments:
  File "/Library/

In [2]:
import os
from collections import defaultdict

from PIL import Image
import pillow_heif

# Data Exploration

In [3]:
data_dir = "./wildsnap-copy"

### Count total image in each subfolder

In [4]:
summary = defaultdict(lambda: defaultdict(int))

total_images = 0
for class_name in sorted(os.listdir(data_dir)):
    class_path = os.path.join(data_dir, class_name)
    if os.path.isdir(class_path):
        for subfolder in sorted(os.listdir(class_path)):
            subfolder_path = os.path.join(class_path, subfolder)
            if os.path.isdir(subfolder_path):
                for file in os.listdir(subfolder_path):
                    summary[class_name][subfolder] += 1
                    total_images += 1


print("Dataset Summary:\n")

print("Total", total_images, "Images\n")

for class_name, subfolders in summary.items():
    total = sum(subfolders.values())
    print(f"{class_name.capitalize():<10} : {total} images")
    for sub_name, count in sorted(subfolders.items()):
        print(f"  - {sub_name:<10} : {count}")
    print()


Dataset Summary:

Total 754 Images

Birds      : 435 images
  - นกกระตั้วหงอนเหลือง : 52
  - นกกระเรียนหงอนพู่ : 43
  - นกซีบร้าฟินซ์ : 3
  - นกพิราบบูดาเปสต์ : 22
  - นกฟินซ์เจ็ดสี : 25
  - นกหงส์หยก  : 1
  - นกเขาใหญ่  : 3
  - นกแก้วกรีนซีค : 23
  - นกแก้วค๊อกคาทีล : 7
  - นกแก้วบลูแอนด์โกลด์มาคอว์ : 47
  - นกแก้วฟอฟัส : 1
  - นกแก้วริงเน็ค(เหลือง) : 23
  - นกแก้วสคาเล็ตมาคอว์ : 38
  - นกแก้วเทาแอฟริกัน : 26
  - นกแก้วเรนโบว์ : 7
  - นกแก้วแบล็คเฮดไคท์ : 22
  - นกแก้วไวท์เบลลี่คีก : 31
  - เป็ดเทศ    : 38
  - ไก่จุก     : 23

Mammals    : 120 images
  - กระต่าย    : 1
  - กระต่าย_jpg : 1
  - กระรอกสวน  : 18
  - กระรอกสวน_jpg : 9
  - ดอร์เมาส์  : 4
  - ลิงกระรอก  : 12
  - ลิงกระรอก_jpg : 8
  - เมียร์แคท  : 43
  - เมียร์แคท_jpg : 14
  - แมววิเชียรมาศ : 10

Reptiles   : 183 images
  - งูหลาม     : 21
  - งูหลามทอง  : 16
  - งูหลามทอง_jpg : 2
  - งูหลามบอล  : 31
  - งูเขียวปากจิ้งจก : 3
  - จระเข้ไคแมนแคระ : 5
  - จระเข้ไคแมนแคระ_jpg : 1
  - จระเข้ไทย (ขาว) : 25
  - จระเข้ไทย (ขาว)_jpg :

### Count all file extensions 

In [5]:
extension_counts = defaultdict(int)

for root, dirs, files in os.walk(data_dir):
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        if ext == '':
            ext = '[no extension]'
        extension_counts[ext] += 1

print("File Extensions Summary:\n")
total_files = 0
for ext, count in sorted(extension_counts.items(), key=lambda x: -x[1]):
    print(f"{ext:<15} : {count} files")
    total_files += count

print(f"\nTotal files: {total_files}")

File Extensions Summary:

.jpg            : 694 files
.heic           : 155 files
[no extension]  : 5 files

Total files: 854


# Pre-processing

### Filter subfolder > 12 images

In [6]:
summary = defaultdict(lambda: defaultdict(int))

for class_name in sorted(os.listdir(data_dir)):
    class_path = os.path.join(data_dir, class_name)
    if os.path.isdir(class_path):
        for subfolder in sorted(os.listdir(class_path)):
            subfolder_path = os.path.join(class_path, subfolder)
            if os.path.isdir(subfolder_path):
                count = 0
                for file in os.listdir(subfolder_path):
                    count += 1
                if count > 12:
                    summary[class_name][subfolder] = count


total_images = sum(
    count for class_data in summary.values() for count in class_data.values()
)

print(f"Total {total_images} Images\n")

for class_name, subfolders in summary.items():
    class_total = sum(subfolders.values())
    print(f"{class_name:<10} : {class_total} images")
    for sub_name, count in subfolders.items():
        print(f"  - {sub_name:<30} : {count}")
    print()

Total 646 Images

Birds      : 413 images
  - นกกระตั้วหงอนเหลือง            : 52
  - นกกระเรียนหงอนพู่              : 43
  - นกพิราบบูดาเปสต์               : 22
  - นกฟินซ์เจ็ดสี                  : 25
  - นกแก้วกรีนซีค                  : 23
  - นกแก้วบลูแอนด์โกลด์มาคอว์      : 47
  - นกแก้วริงเน็ค(เหลือง)          : 23
  - นกแก้วสคาเล็ตมาคอว์            : 38
  - นกแก้วเทาแอฟริกัน              : 26
  - นกแก้วแบล็คเฮดไคท์             : 22
  - นกแก้วไวท์เบลลี่คีก            : 31
  - เป็ดเทศ                        : 38
  - ไก่จุก                         : 23

Mammals    : 75 images
  - กระรอกสวน                      : 18
  - เมียร์แคท                      : 43
  - เมียร์แคท_jpg                  : 14

Reptiles   : 158 images
  - งูหลาม                         : 21
  - งูหลามทอง                      : 16
  - งูหลามบอล                      : 31
  - จระเข้ไทย (ขาว)                : 25
  - จิ้งเหลนสีน้ำเงิน              : 16
  - ตุ๊กแกเขียวยักษ์มาดากัสการ์    : 19
  - อีกัวน่า                 

### Convert .heic to .jpg

In [7]:
def convert_heic_to_jpg_inplace(data_dir, delete_original=False):
    for class_name in os.listdir(data_dir):
        class_path = os.path.join(data_dir, class_name)
        if os.path.isdir(class_path):
            for subfolder in os.listdir(class_path):
                subfolder_path = os.path.join(class_path, subfolder)
                if os.path.isdir(subfolder_path):
                    for filename in os.listdir(subfolder_path):
                        if filename.lower().endswith('.heic'):
                            heic_path = os.path.join(subfolder_path, filename)
                            jpg_filename = os.path.splitext(filename)[0] + '.jpg'
                            jpg_path = os.path.join(subfolder_path, jpg_filename)

                            # อ่านไฟล์ HEIC
                            heif_file = pillow_heif.read_heif(heic_path)
                            image = Image.frombytes(
                                heif_file.mode,
                                heif_file.size,
                                heif_file.data,
                                "raw"
                            )

                            image.save(jpg_path, "JPEG", quality=95)
                            print(f"✅ Converted {filename} → {jpg_filename}")

                            if delete_original:
                                os.remove(heic_path)
                                print(f"🗑️ Deleted original: {filename}")

convert_heic_to_jpg_inplace(data_dir, delete_original=True)


✅ Converted 20250423_040220017_iOS.heic → 20250423_040220017_iOS.jpg
🗑️ Deleted original: 20250423_040220017_iOS.heic
✅ Converted 20250423_040101833_iOS.heic → 20250423_040101833_iOS.jpg
🗑️ Deleted original: 20250423_040101833_iOS.heic
✅ Converted 20250423_040011429_iOS.heic → 20250423_040011429_iOS.jpg
🗑️ Deleted original: 20250423_040011429_iOS.heic
✅ Converted 20250423_040223106_iOS.heic → 20250423_040223106_iOS.jpg
🗑️ Deleted original: 20250423_040223106_iOS.heic
✅ Converted 20250423_042230419_iOS.heic → 20250423_042230419_iOS.jpg
🗑️ Deleted original: 20250423_042230419_iOS.heic
✅ Converted 20250423_042231049_iOS.heic → 20250423_042231049_iOS.jpg
🗑️ Deleted original: 20250423_042231049_iOS.heic
✅ Converted 20250423_042219126_iOS.heic → 20250423_042219126_iOS.jpg
🗑️ Deleted original: 20250423_042219126_iOS.heic
✅ Converted 20250423_042228245_iOS.heic → 20250423_042228245_iOS.jpg
🗑️ Deleted original: 20250423_042228245_iOS.heic
✅ Converted 20250423_042229683_iOS.heic → 20250423_04222

### Check Extension Summary Again

In [9]:
extension_counts = defaultdict(int)

for root, dirs, files in os.walk(data_dir):
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        if ext == '':
            ext = '[no extension]'
        extension_counts[ext] += 1

print("File Extensions Summary:\n")
total_files = 0
for ext, count in sorted(extension_counts.items(), key=lambda x: -x[1]):
    print(f"{ext:<15} : {count} files")
    total_files += count

print(f"\nTotal files: {total_files}")

File Extensions Summary:

.jpg            : 849 files
[no extension]  : 5 files

Total files: 854
